In [3]:
import os
import sys
import requests
from pathlib import Path
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torchvision.models import resnet18
from safetensors.torch import load_file
import pandas as pd
import torch.nn.functional as F
from tqdm import tqdm

In [4]:
API_KEY = "YOUR_API_KEY_HERE"
SCRIPT_DIR = os.getcwd()
print(SCRIPT_DIR)


TARGET_CKPT = os.path.join(SCRIPT_DIR, "target_model", "weights.safetensors")
SUSPECT_DIR = os.path.join(SCRIPT_DIR, "suspect_models")
DATA_ROOT = os.path.join(SCRIPT_DIR, "cifar100_data")
NUM_SUSPECTS = 360
BATCH_SIZE = 256
NUM_BATCHES = 40

c:\Users\Varun\OneDrive\Desktop\Saarland University\Semester 1\TML\tutorial\tml_task2\tml26_task2


In [5]:

def make_model():
    model = resnet18(weights=None)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, 100)
    return model


def get_dataloader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408), 
                             (0.2675, 0.2565, 0.2761)),
    ])
    
    # Using the test set for unbiased functional signature comparison
    dataset = datasets.CIFAR100(root=DATA_ROOT, train=False, download=True, transform=transform)
    loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
    return loader

In [6]:
def get_model_logits(model, dataloader, device, num_batches):

    model.eval()
    all_logits = []
    
    with torch.no_grad():
        for i, (images, _) in enumerate(dataloader):
            if i >= num_batches:
                break
            images = images.to(device)
            logits = model(images)
            all_logits.append(logits.cpu())
            
    return torch.cat(all_logits, dim=0)

In [ ]:
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    dataloader = get_dataloader()
    target_model = make_model()
    target_model.load_state_dict(load_file(TARGET_CKPT, device="cpu"), strict=True)
    target_model.to(device)
    
    
    target_logits = get_model_logits(target_model, dataloader, device, NUM_BATCHES)
    target_flat = target_logits.view(-1) 


    subset_ids = list(range(NUM_SUSPECTS))
    confidence_scores = []

    for model_id in tqdm(subset_ids):
        suspect_model = make_model()
        
        
        suspect_ckpt_path = os.path.join(SUSPECT_DIR, f"suspect_{model_id:03d}.safetensors")    
        suspect_model.load_state_dict(load_file(suspect_ckpt_path, device="cpu"), strict=True)
        suspect_model.to(device)
        
        suspect_logits = get_model_logits(suspect_model, dataloader, device, NUM_BATCHES)
        suspect_flat = suspect_logits.view(-1)

        # Temperature=2
        # target_probs = F.softmax(target_flat/Temperature, dim=0)
        # suspect_probs = F.softmax(suspect_flat/Temperature, dim=0)
        # score = F.cosine_similarity(target_probs, suspect_probs, dim=0).item()
        # score= F.cosine_similarity(target_flat, suspect_flat, dim=0).item()
        # Clamp negative scores to 0

        target_probs = F.softmax(target_logits, dim=1)
        suspect_log_probs = F.log_softmax(suspect_logits , dim=1)
        kl_distance = F.kl_div(suspect_log_probs, target_probs, reduction='batchmean').item()
        score = 1.0 / (1.0 + kl_distance)
        score = max(0.0, score)
        confidence_scores.append(score)


    submission_df = pd.DataFrame({
        "id": subset_ids,
        "score": confidence_scores
    })
    submission_path = os.path.join(SCRIPT_DIR, "submissions", "submission.csv")
    submission_df.to_csv(submission_path, index=None)
    print(f"\nEvaluation Complete! Submission saved to {submission_path}.")
    
    

In [11]:
if __name__ == "__main__":
    main()

Using device: cuda


c:\Users\Varun\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Extracting Target Model Logits...
Evaluating Suspect Models...


100%|██████████| 360/360 [43:29<00:00,  7.25s/it]    


Evaluation Complete! Submission saved to c:\Users\Varun\OneDrive\Desktop\Saarland University\Semester 1\TML\tutorial\tml_task2\tml26_task2\submissions\submission.csv.
Sample scores:
   id     score
0   0  0.733548
1   1  0.605466
2   2  0.947227
3   3  0.639385
4   4  1.000000


In [9]:
print(NUM_BATCHES)

40
